# L60 · 自动求梯度 — autograd 机制

**目标**：用 PyTorch autograd 对同一个计算图求梯度，并与 `a3_backward.ipynb` 中手写的 `Value` 引擎数值比对，确认两套机制在数值上完全等价。

🔗 **Aurora 连接**：Month 2–5 的所有模型训练都调用 `.backward()`；本节确立的等价性是后续用 torch 替换手写 Value 图的信心基础。具体入口：`src/aurora/` 各核的 `trainer.py`（Month 3 起引入）中每个 `loss.backward()` 调用均依赖此机制。

PyTorch 的自动微分与 `a3_backward.ipynb` 的 `Value` 类做同一件事：在正向传播时把操作记录为有向无环图，反向传播时沿边累积链式法则。
两者的核心差异在于 PyTorch 用 C++ 核心实现、支持批量张量，而我们的 `Value` 用纯 Python 展示了同样的拓扑排序 + 反向钉子逻辑。
本节用具体数值让两者“对表”，从而建立直觉：autograd 没有魔法，只是高效的反向图遍历。

In [ ]:
import torch
import numpy as np

## 1. 计算图：`requires_grad=True` 与叶 / 非叶节点

当张量带 `requires_grad=True` 时，每次运算会生成一个 `grad_fn` 并把结果节点挂在图上。
**叶节点（leaf tensor）**：由用户直接创建，`is_leaf == True`，梯度会在 `.backward()` 后存入 `.grad`。
**非叶节点**：由运算派生，`is_leaf == False`，默认不保留梯度（`.grad` 为 `None`），需要 `retain_grad()` 才能查看。

```
a (leaf)  b (leaf)
  \          /
   c = a * b   <- c.grad_fn = MulBackward0, c.is_leaf = False
       |
   L = c + a   <- L.grad_fn = AddBackward0
```

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
L = c + a

print('a.is_leaf:', a.is_leaf)   # True
print('c.is_leaf:', c.is_leaf)   # False
print('c.grad_fn:', c.grad_fn)   # MulBackward0
print('L.grad_fn:', L.grad_fn)   # AddBackward0

## 2. `.detach()` 与 `.grad` 的存储规则

`.detach()` 返回一个与原张量共享数据但**切断梯度流**的新张量——之后的运算不再记录到图中。
常见用途：把中间激活値转为纯 numpy 打印，或在 GAN 训练中固定某一路径。

`.grad` 仅为叶节点自动保留；对非叶节点必须在 `backward()` 之前调用 `retain_grad()`，否则 `.grad` 保持 `None`。

```python
# 切断示例
x = torch.tensor(1.0, requires_grad=True)
y = x * 2
z = y.detach() * 3   # z 不在图上
z.backward()          # 报错：z 没有 grad_fn
```

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x * 2          # 非叶节点
y.retain_grad()    # 开启非叶梯度保留
L2 = y * 3
L2.backward()

print('x.grad:', x.grad)   # 6.0  (dL2/dx = 3*2)
print('y.grad:', y.grad)   # 3.0  (dL2/dy = 3)

# detach 演示
y_np = y.detach().numpy()   # 不影响图
print('y_np (detach):', y_np)

## 3. 梯度累积与 `zero_grad()`

每次调用 `.backward()`，PyTorch 会把新梯度**加到** `.grad` 上，而不是覆盖。
这是有意设计（支持梯度累积训练技巧），但在标准训练循环中会导致梯度错误地叠加。

规范做法：
```python
for batch in dataloader:
    optimizer.zero_grad()   # 清零上一步残留
    loss = model(batch)
    loss.backward()
    optimizer.step()
```

手写 `Value` 引擎同理：每次反向前需手动把所有节点的 `.grad` 归零。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

# 两次 backward，不清零
for i in range(2):
    L3 = w * 3.0
    L3.backward()
    print(f'第 {i+1} 次 backward 后 w.grad = {w.grad}')  # 3.0, 6.0

# 清零后再做
w.grad.zero_()
L3 = w * 3.0
L3.backward()
print('zero_grad 后 w.grad =', w.grad)  # 3.0

## 4. ✏️ 实现 `verify_gradients()`

**推理路线**：
1. 用 `torch.tensor` 创建 `a=2.0, b=3.0`，均设 `requires_grad=True`
2. 构造 `L = a*b*b + a`（与 `a3_backward.ipynb` 中完全相同的表达式）
3. 调用 `L.backward()`
4. 读取 `a.grad`、`b.grad`，与解析导数 `dL/da = b^2 = 9`、`dL/db = 2ab = 12` 比较
5. 同时与 `Value` 引擎结果对比，打印两者差値（应为 0）

**参考输入输出**：
```
a = 2.0, b = 3.0
L = a*b*b + a = 2*9 + 2 = 20.0
dL/da = b^2 = 9.0
dL/db = 2*a*b = 12.0
```

<details><summary>点击查看参考实现</summary>

```python
def verify_gradients():
    a = torch.tensor(2.0, requires_grad=True)
    b = torch.tensor(3.0, requires_grad=True)
    L = a * b * b + a
    L.backward()
    da_torch = a.grad.item()
    db_torch = b.grad.item()
    da_ref = 3.0**2          # b^2 = 9.0
    db_ref = 2 * 2.0 * 3.0  # 2ab = 12.0
    print(f'torch  dL/da={da_torch:.4f}  dL/db={db_torch:.4f}')
    print(f'解析値 dL/da={da_ref:.4f}    dL/db={db_ref:.4f}')
    print(f'误差   dL/da={abs(da_torch-da_ref):.2e}  dL/db={abs(db_torch-db_ref):.2e}')
    return da_torch, db_torch
```

</details>

In [ ]:
def verify_gradients():
    # ✏️ TODO: 创建 a=2.0, b=3.0 (requires_grad=True)
    # ✏️ TODO: 构造 L = a*b*b + a
    # ✏️ TODO: 调用 L.backward()
    # ✏️ TODO: 打印 a.grad, b.grad 及与解析値的误差
    # ✏️ TODO: return a.grad.item(), b.grad.item()
    pass

verify_gradients()

In [ ]:
# --- 检查 ---
def _ref_verify():
    a = torch.tensor(2.0, requires_grad=True)
    b = torch.tensor(3.0, requires_grad=True)
    L = a * b * b + a
    L.backward()
    return a.grad.item(), b.grad.item()

da, db = _ref_verify()
assert abs(da - 9.0) < 1e-5,  f'dL/da 应为 9.0，实际 {da}'
assert abs(db - 12.0) < 1e-5, f'dL/db 应为 12.0，实际 {db}'
print('✅ dL/da =', da, '| dL/db =', db, '— 与解析値一致')

## 5. 参数实验：自制 Value vs torch 对 `L = tanh(w*x + b)` 的梯度误差

**实验参数**：`w` 在 `[-2, -1, 0, 1, 2]` 上扫描，`x=1.0, b=0.5` 固定。

**预期现象**：
- `atol` 级别误差应 < 1e-6（浮点精度以内）
- `tanh` 饱和区（`|w|` 大）梯度趋近 0，两者差値绝对値也趋近 0
- 图中两条曲线（torch / Value）完全重叠，误差曲线几乎贴近 x 轴

In [ ]:
import matplotlib.pyplot as plt
import math

# ---- 手写 Value（最小实现，与 a3_backward.ipynb 一致）----
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _back():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _back
        return out

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _back():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _back
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _back():
            self.grad += (1 - t**2) * out.grad
        out._backward = _back
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()

# ---- 对比实验 ----
ws = [-2.0, -1.0, 0.0, 1.0, 2.0]
x_val, b_val = 1.0, 0.5

grads_torch, grads_value, errors = [], [], []

for w_val in ws:
    # torch
    wt = torch.tensor(w_val, requires_grad=True)
    xt = torch.tensor(x_val)
    bt = torch.tensor(b_val)
    Lt = torch.tanh(wt * xt + bt)
    Lt.backward()
    gt = wt.grad.item()
    grads_torch.append(gt)

    # Value
    wv = Value(w_val)
    xv = Value(x_val)
    bv = Value(b_val)
    Lv = (wv * xv + bv).tanh()
    Lv.backward()
    gv = wv.grad
    grads_value.append(gv)

    errors.append(abs(gt - gv))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(ws, grads_torch, 'o-', label='torch autograd', linewidth=2)
ax1.plot(ws, grads_value, 's--', label='手写 Value', linewidth=2, alpha=0.7)
ax1.set_xlabel('w')
ax1.set_ylabel('dL/dw')
ax1.set_title('dL/dw for L=tanh(w*x+b)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(ws, [max(e, 1e-20) for e in errors], 'r^-', linewidth=2)
ax2.set_xlabel('w')
ax2.set_ylabel('|torch - Value| (log scale)')
ax2.set_title('梯度误差（atol 级别）')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('最大误差 atol:', max(errors))

## 小结

`verify_gradients()` 输出了 `dL/da=9.0, dL/db=12.0`，与手写 `Value` 引擎及解析导数在 1e-6 精度内吴合。
本节确认：PyTorch `autograd` 的计算图遍历、叶节点梯度存储、以及 `zero_grad()` 规范，与 `a3_backward.ipynb` 的纯 Python 实现在数学上等价。
这两套梯度引擎共同支撑 Aurora 的 `src/aurora/` 各核训练循环——Month 3 起的 `trainer.py` 将直接调用 `loss.backward()`。
下一节（M2-P3）在此基础上构建第一个真正的线性层，用 `nn.Module` 封装参数并接入 `optimizer.step()`。